# Descargar los PDF de varias páginas de la UC3M en carpetas separadas

Este cuaderno está preparado para:

1. Recorrer varias páginas de la UC3M.
2. Extraer todos los enlaces que apunten a **PDF** o a **Google Drive** con archivos PDF.
3. Crear una carpeta distinta para cada página.
4. Descargar los documentos en su carpeta correspondiente.
5. Generar informes por carpeta y un informe global final.


In [ ]:
# Configuración
PAGES = [
    {
        "nombre": "Historia de la Filosofía",
        "url": "https://www.uc3m.es/ss/Satellite/evau/es/TextoMixta/1371318332067/",
        "output_dir": "filosofia",
        "procesar": False
    },
    {
        "nombre": "Física",
        "url": "https://www.uc3m.es/ss/Satellite/evau/es/TextoMixta/1371318299061/",
        "output_dir": "fisica",
        "procesar": False
    },
    {
        "nombre": "Inglés II",
        "url": "https://www.uc3m.es/ss/Satellite/evau/es/TextoMixta/1371318276778/",
        "output_dir": "ingles",
        "procesar": False
    },
    {
        "nombre": "Lengua Castellana y Literatura",
        "url": "https://www.uc3m.es/ss/Satellite/evau/es/TextoMixta/1371318275798/",
        "output_dir": "lengua",
        "procesar": False
    },
    {
        "nombre": "Matemáticas II",
        "url": "https://www.uc3m.es/ss/Satellite/evau/es/TextoMixta/1371318154983/",
        "output_dir": "matematicas",
        "procesar": False
    },
    {
        "nombre": "Quimica",
        "url": "https://www.uc3m.es/ss/Satellite/evau/es/TextoMixta/1371318182329/",
        "output_dir": "quimica",
        "procesar": False
    },
    {
        "nombre": "Tecnología",
        "url": "https://www.uc3m.es/ss/Satellite/evau/es/TextoMixta/1371377314206/",
        "output_dir": "tecnologia",
        "procesar": False
    },
]

TIMEOUT = 30
OVERWRITE = False   # Cambia a True si quieres volver a descargar ficheros ya existentes
SLEEP_BETWEEN_REQUESTS = 0.3

In [2]:
# Instalación opcional de dependencias si hiciera falta
import sys
import subprocess
import importlib

required_packages = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Instalando {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

In [3]:
# Imports
import os
import re
import json
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, unquote

import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

## Funciones auxiliares

In [4]:
def make_session():
    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/123.0 Safari/537.36"
        )
    })
    return session


def sanitize_filename(name: str, default: str = "documento.pdf") -> str:
    if not name:
        return default
    name = unquote(str(name)).strip()
    name = re.sub(r'[\\/:*?"<>|]+', "_", name)
    name = re.sub(r"\s+", " ", name).strip(" .")
    if not name.lower().endswith(".pdf"):
        name += ".pdf"
    return name or default


def content_disposition_filename(headers) -> str | None:
    cd = headers.get("content-disposition", "")
    if not cd:
        return None

    match_star = re.search(r"filename\*\s*=\s*UTF-8''([^;]+)", cd, flags=re.I)
    if match_star:
        return sanitize_filename(unquote(match_star.group(1)))

    match = re.search(r'filename\s*=\s*"([^"]+)"', cd, flags=re.I)
    if match:
        return sanitize_filename(match.group(1))

    match_simple = re.search(r"filename\s*=\s*([^;]+)", cd, flags=re.I)
    if match_simple:
        return sanitize_filename(match_simple.group(1).strip())

    return None


def get_google_drive_file_id(url: str) -> str | None:
    parsed = urlparse(url)
    hostname = parsed.netloc.lower()

    if "drive.google.com" not in hostname and "docs.google.com" not in hostname:
        return None

    match = re.search(r"/file/d/([^/]+)", parsed.path)
    if match:
        return match.group(1)

    qs = parse_qs(parsed.query)
    if "id" in qs and qs["id"]:
        return qs["id"][0]

    return None


def classify_link(url: str) -> str:
    lower = url.lower()

    if lower.endswith(".pdf"):
        return "pdf"

    file_id = get_google_drive_file_id(url)
    if file_id:
        return "gdrive"

    return "other"


def extract_pdf_links_from_page(page_url: str, page_name: str) -> list[dict]:
    session = make_session()
    response = session.get(page_url, timeout=TIMEOUT)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    found = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", "").strip()
        text = a.get_text(" ", strip=True)
        if not href:
            continue

        absolute_url = urljoin(page_url, href)
        link_type = classify_link(absolute_url)

        if link_type == "other":
            continue

        found.append({
            "pagina": page_name,
            "texto": text or "Sin texto",
            "url": absolute_url,
            "tipo": link_type,
            "file_id": get_google_drive_file_id(absolute_url),
        })

    dedup = {}
    for item in found:
        key = f"gdrive::{item['file_id']}" if item["file_id"] else f"url::{item['url']}"
        if key not in dedup:
            dedup[key] = item

    return list(dedup.values())


def try_direct_pdf_download(session: requests.Session, url: str, dest_path: Path) -> tuple[bool, str, str]:
    with session.get(url, stream=True, timeout=TIMEOUT, allow_redirects=True) as r:
        r.raise_for_status()

        content_type = (r.headers.get("content-type") or "").lower()
        filename = content_disposition_filename(r.headers) or dest_path.name

        if "pdf" not in content_type and not filename.lower().endswith(".pdf"):
            return False, f"No parece un PDF. content-type={content_type or 'desconocido'}", dest_path.name

        final_path = dest_path.with_name(sanitize_filename(filename, dest_path.name))

        if final_path.exists() and not OVERWRITE:
            return True, f"Ya existe: {final_path.name}", final_path.name

        with open(final_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 128):
                if chunk:
                    f.write(chunk)

        return True, f"Descargado: {final_path.name}", final_path.name


def google_drive_download_candidates(file_id: str) -> list[str]:
    return [
        f"https://drive.google.com/uc?export=download&id={file_id}",
        f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t",
    ]


def download_google_drive_file(session: requests.Session, file_id: str, dest_path: Path) -> tuple[bool, str, str]:
    last_error = "No se pudo descargar desde Google Drive"

    for candidate in google_drive_download_candidates(file_id):
        try:
            with session.get(candidate, stream=True, timeout=TIMEOUT, allow_redirects=True) as r:
                r.raise_for_status()

                content_type = (r.headers.get("content-type") or "").lower()
                filename = content_disposition_filename(r.headers) or dest_path.name
                final_path = dest_path.with_name(sanitize_filename(filename, dest_path.name))

                if "text/html" in content_type and "pdf" not in content_type:
                    html = b""
                    for chunk in r.iter_content(chunk_size=8192):
                        html += chunk
                        if len(html) > 200000:
                            break
                    html_text = html.decode("utf-8", errors="ignore")

                    match = re.search(r'href="(/uc\?export=download[^"]+confirm=[^"]+)"', html_text)
                    if match:
                        confirm_url = urljoin("https://drive.google.com", match.group(1).replace("&amp;", "&"))
                        with session.get(confirm_url, stream=True, timeout=TIMEOUT, allow_redirects=True) as r2:
                            r2.raise_for_status()
                            filename = content_disposition_filename(r2.headers) or dest_path.name
                            final_path = dest_path.with_name(sanitize_filename(filename, dest_path.name))

                            if final_path.exists() and not OVERWRITE:
                                return True, f"Ya existe: {final_path.name}", final_path.name

                            with open(final_path, "wb") as f:
                                for chunk2 in r2.iter_content(chunk_size=1024 * 128):
                                    if chunk2:
                                        f.write(chunk2)

                            return True, f"Descargado: {final_path.name}", final_path.name

                    last_error = f"Google Drive devolvió HTML en lugar de PDF ({candidate})"
                    continue

                if final_path.exists() and not OVERWRITE:
                    return True, f"Ya existe: {final_path.name}", final_path.name

                with open(final_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 128):
                        if chunk:
                            f.write(chunk)

                return True, f"Descargado: {final_path.name}", final_path.name

        except Exception as e:
            last_error = str(e)

    return False, last_error, dest_path.name


def suggested_filename(item: dict, index: int) -> str:
    text = item.get("texto") or f"documento_{index:03d}"
    return sanitize_filename(text, default=f"documento_{index:03d}.pdf")

## Extraer enlaces de las tres páginas

In [5]:
all_links = []
page_summaries = []

for page in PAGES:
    if page["procesar"] is True:
        print(f"Procesando página '{page['nombre']}'")
        links = extract_pdf_links_from_page(page["url"], page["nombre"])
        for item in links:
            item["output_dir"] = page["output_dir"]
        all_links.extend(links)

        page_summaries.append({
            "pagina": page["nombre"],
            "url": page["url"],
            "output_dir": page["output_dir"],
            "enlaces_detectados": len(links),
        })
    else:
        print(f"Omitiendo página '{page['nombre']}' (procesar=False)")
    
df_pages = pd.DataFrame(page_summaries)
df_links = pd.DataFrame(all_links)

print("Resumen por página:")
display(df_pages)

print(f"Total de enlaces detectados: {len(df_links)}")
display(df_links)

Omitiendo página 'Historia de la Filosofía' (procesar=False)
Procesando página 'Física'
Omitiendo página 'Inglés II' (procesar=False)
Omitiendo página 'Lengua Castellana y Literatura' (procesar=False)
Procesando página 'Matemáticas II'
Procesando página 'Quimica'
Procesando página 'Tecnología'
Resumen por página:


,pagina,url,output_dir,enlaces_detectados
0,Física,https://www.uc3m.es/ss/Satellite/evau/es/Texto...,fisica,29
1,Matemáticas II,https://www.uc3m.es/ss/Satellite/evau/es/Texto...,matematicas,33
2,Quimica,https://www.uc3m.es/ss/Satellite/evau/es/Texto...,quimica,29
3,Tecnología,https://www.uc3m.es/ss/Satellite/evau/es/Texto...,tecnologia,11


Total de enlaces detectados: 102


,pagina,texto,url,tipo,file_id,output_dir
0,Física,Física 2025-2026,https://drive.google.com/file/d/18qaoS5fMfycdZ...,gdrive,18qaoS5fMfycdZbLl5RggFsh9YWs1kNOw,fisica
1,Física,Física 2024-2025,https://drive.google.com/file/d/1TyVVdwUC7HePl...,gdrive,1TyVVdwUC7HePlJBkR_FRDlWGXScaX39i,fisica
2,Física,Física 2023-2024,https://drive.google.com/file/d/1EC0L7aRLD6xM6...,gdrive,1EC0L7aRLD6xM6JZKxMSyFOnNt8QFHr3w,fisica
3,Física,Física 2022-2023,https://drive.google.com/file/d/1qT5Mt4maBT9UD...,gdrive,1qT5Mt4maBT9UDZJcu2WTomrBPqmQ5MGM,fisica
4,Física,Física 2021-2022,https://drive.google.com/file/d/1i_WlSWSe1ibi9...,gdrive,1i_WlSWSe1ibi9l360h0fqiTIsDuhEbcC,fisica
...,...,...,...,...,...,...
97,Tecnología,Tecnología e Ingeniería II 2024-2025 coinciden...,https://drive.google.com/file/d/13oXuBH_5fDjYo...,gdrive,13oXuBH_5fDjYoWJVbxABvIR6inhOJ_kQ,tecnologia
98,Tecnología,Tecnología e Ingeniería II 2024-2025,https://drive.google.com/file/d/1mWWmXE7dNSYJ5...,gdrive,1mWWmXE7dNSYJ5nsKrcO2fhisnCq2ze3d,tecnologia
99,Tecnología,Tecnología e Ingeniería II 2023-2024,https://drive.google.com/file/d/1YMHpN_xdaqpP1...,gdrive,1YMHpN_xdaqpP1K9MbYcZFvQCrMIuXSVc,tecnologia
100,Tecnología,Criterios de Corrección Generales 2025-2026,https://drive.google.com/file/d/1oJNgNfhuKd6Hf...,gdrive,1oJNgNfhuKd6HfMJWiy19O-6nrj4rvj6h,tecnologia


## Descargar los PDF en carpetas diferentes

In [6]:
session = make_session()
results = []

for page in PAGES:
    output_path = Path(page["output_dir"])
    output_path.mkdir(parents=True, exist_ok=True)

for i, item in enumerate(all_links, start=1):
    page_name = item["pagina"]
    output_path = Path(item["output_dir"])
    url = item["url"]
    link_type = item["tipo"]
    file_id = item.get("file_id")
    filename = suggested_filename(item, i)
    dest_path = output_path / filename

    print(f"[{i}/{len(all_links)}] {page_name} -> {item['texto']}")

    try:
        if link_type == "pdf":
            ok, message, saved_name = try_direct_pdf_download(session, url, dest_path)
        elif link_type == "gdrive":
            ok, message, saved_name = download_google_drive_file(session, file_id, dest_path)
        else:
            ok, message, saved_name = False, f"Tipo de enlace no soportado: {link_type}", dest_path.name

    except Exception as e:
        ok, message, saved_name = False, str(e), dest_path.name

    results.append({
        "pagina": page_name,
        "texto": item["texto"],
        "tipo": link_type,
        "url": url,
        "output_dir": str(output_path),
        "archivo": saved_name,
        "ok": ok,
        "mensaje": message,
    })

    time.sleep(SLEEP_BETWEEN_REQUESTS)

df_results = pd.DataFrame(results)
display(df_results)

[1/102] Física -> Física 2025-2026
[2/102] Física -> Física 2024-2025
[3/102] Física -> Física 2023-2024
[4/102] Física -> Física 2022-2023
[5/102] Física -> Física 2021-2022
[6/102] Física -> Física 2020-2021
[7/102] Física -> Física 2019-2020
[8/102] Física -> Física  2018-2019
[9/102] Física -> Física 2017-2018
[10/102] Física -> Física 2024-2025
[11/102] Física -> Física 2023-2024
[12/102] Física -> Física 2022-2023 coincidencias
[13/102] Física -> Física 2022-2023
[14/102] Física -> Física 2021-2022
[15/102] Física -> Física 2020-2021
[16/102] Física -> Física 2020-2021 coincidencias
[17/102] Física -> Física 2019-2020
[18/102] Física -> Física 2018-2019
[19/102] Física -> Física 2017-2018
[20/102] Física -> Física 2024-2025
[21/102] Física -> Física 2023-2024
[22/102] Física -> Física 2022-2023
[23/102] Física -> Física 2021-2022
[24/102] Física -> Física 2020-2021
[25/102] Física -> Física 2019-2020
[26/102] Física -> Física 2018-2019
[27/102] Física -> Física 2017-2018
[28/102]

,pagina,texto,tipo,url,output_dir,archivo,ok,mensaje
0,Física,Física 2025-2026,gdrive,https://drive.google.com/file/d/18qaoS5fMfycdZ...,fisica,2025-2026 Modelo examen FÃ­sica.pdf,True,Descargado: 2025-2026 Modelo examen FÃ­sica.pdf
1,Física,Física 2024-2025,gdrive,https://drive.google.com/file/d/1TyVVdwUC7HePl...,fisica,2024-2025 Modelo de examen de FÃ­sica.pdf,True,Descargado: 2024-2025 Modelo de examen de FÃ­s...
2,Física,Física 2023-2024,gdrive,https://drive.google.com/file/d/1EC0L7aRLD6xM6...,fisica,2023-2024 Modelo examen FÃ­sica.pdf,True,Descargado: 2023-2024 Modelo examen FÃ­sica.pdf
3,Física,Física 2022-2023,gdrive,https://drive.google.com/file/d/1qT5Mt4maBT9UD...,fisica,2022-2023 modelo examen FÃ­sica.pdf,True,Descargado: 2022-2023 modelo examen FÃ­sica.pdf
4,Física,Física 2021-2022,gdrive,https://drive.google.com/file/d/1i_WlSWSe1ibi9...,fisica,2021-2022 Modelo examen FÃ­sica.pdf,True,Descargado: 2021-2022 Modelo examen FÃ­sica.pdf
...,...,...,...,...,...,...,...,...
97,Tecnología,Tecnología e Ingeniería II 2024-2025 coinciden...,gdrive,https://drive.google.com/file/d/13oXuBH_5fDjYo...,tecnologia,2024-2025 Extraordinaria TecnologÃ­a e Ingenie...,True,Descargado: 2024-2025 Extraordinaria TecnologÃ...
98,Tecnología,Tecnología e Ingeniería II 2024-2025,gdrive,https://drive.google.com/file/d/1mWWmXE7dNSYJ5...,tecnologia,2024-2025 Extraordinaria TecnologÃ­a e Ingenie...,True,Descargado: 2024-2025 Extraordinaria TecnologÃ...
99,Tecnología,Tecnología e Ingeniería II 2023-2024,gdrive,https://drive.google.com/file/d/1YMHpN_xdaqpP1...,tecnologia,2023-2024 Extraordinaria Soluciones TecnologÃ­...,True,Descargado: 2023-2024 Extraordinaria Solucione...
100,Tecnología,Criterios de Corrección Generales 2025-2026,gdrive,https://drive.google.com/file/d/1oJNgNfhuKd6Hf...,tecnologia,2025-2026 Criterios de CorrecciÃ³n Generales.pdf,True,Descargado: 2025-2026 Criterios de CorrecciÃ³n...


## Guardar informes por carpeta y un informe global

In [7]:
root_output = Path("pdf_uc3m")
root_output.mkdir(parents=True, exist_ok=True)

# Informe global
global_json = root_output / "informe_global_descargas.json"
global_csv = root_output / "informe_global_descargas.csv"

with open(global_json, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

df_results.to_csv(global_csv, index=False, encoding="utf-8-sig")

# Informes por carpeta
for page in PAGES:
    page_df = df_results[df_results["pagina"] == page["nombre"]].copy()
    page_output = Path(page["output_dir"])
    page_output.mkdir(parents=True, exist_ok=True)

    page_json = page_output / "informe_descargas.json"
    page_csv = page_output / "informe_descargas.csv"

    with open(page_json, "w", encoding="utf-8") as f:
        json.dump(page_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

    page_df.to_csv(page_csv, index=False, encoding="utf-8-sig")

print(f"Informe global JSON: {global_json}")
print(f"Informe global CSV:  {global_csv}")

Informe global JSON: pdf_uc3m/informe_global_descargas.json
Informe global CSV:  pdf_uc3m/informe_global_descargas.csv


## Resumen final

In [8]:
summary_rows = []

for page in PAGES:
    page_df = df_results[df_results["pagina"] == page["nombre"]]
    total = len(page_df)
    ok_count = int(page_df["ok"].sum()) if total else 0
    fail_count = total - ok_count

    summary_rows.append({
        "pagina": page["nombre"],
        "carpeta": page["output_dir"],
        "detectados": total,
        "descargados_ok": ok_count,
        "fallidos": fail_count,
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

total = len(df_results)
ok_count = int(df_results["ok"].sum()) if total else 0
fail_count = total - ok_count

print(f"Total detectados: {total}")
print(f"Descargados/ok:   {ok_count}")
print(f"Fallidos:         {fail_count}")
print(f"Carpeta raíz:     {root_output.resolve()}")

if fail_count:
    print("\nDescargas fallidas:")
    display(df_results.loc[~df_results["ok"], ["pagina", "texto", "url", "mensaje"]])

,pagina,carpeta,detectados,descargados_ok,fallidos
0,Historia de la Filosofía,filosofia,0,0,0
1,Física,fisica,29,29,0
2,Inglés II,ingles,0,0,0
3,Lengua Castellana y Literatura,lengua,0,0,0
4,Matemáticas II,matematicas,33,33,0
5,Quimica,quimica,29,29,0
6,Tecnología,tecnologia,11,11,0


Total detectados: 102
Descargados/ok:   102
Fallidos:         0
Carpeta raíz:     /home/adolfo/Proyectos/PAU-ALBA/pdf_uc3m


## Notas

- El cuaderno detecta tanto enlaces **PDF directos** como enlaces de **Google Drive**.
- Crea una carpeta raíz llamada `pdf_uc3m` y dentro genera **tres subcarpetas**, una por cada página.
- Si algún archivo no se descarga, suele deberse a permisos, cambios en Google Drive o protección adicional del enlace.
- Si quieres rehacer todas las descargas desde cero, cambia `OVERWRITE = True`.
- Puedes añadir más páginas repitiendo el mismo formato dentro de `PAGES`.